In [ ]:
import warnings
warnings.filterwarnings("ignore") 

import os
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import plotly.express as px

import pandas as pd
import joblib
from lightgbm import LGBMRegressor

from adjustText import adjust_text

import math
from matplotlib.ticker import MultipleLocator

import seaborn as sns
from scipy.stats import ks_2samp 
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    root_mean_squared_error, 
    r2_score, 
    max_error, 
    explained_variance_score
)

DFT_SOURCES = ['alexandria_2d', '2dmatpedia', 'c2db', 'mc2d']
SOURCES = ['gbfs_pred', DFT_SOURCES[0], DFT_SOURCES[1], DFT_SOURCES[2], DFT_SOURCES[3]]

# Define file paths
cwd = os.getcwd()
file_path = f"{cwd}\\pkl"
final_file_path = f"{file_path}\\bandgap_results\\final"

features_list_file_path = second_features_list = f"{file_path}\\bandgap_results\\features_selected_from_RFE_bandgap.pkl"

model_file_path = f"{final_file_path}\\2d_combined_bandgap_final_model.pkl"

expt_data_file_path = f"{final_file_path}\\expt_band_gap_dir.xlsx"
dft_data_file_path = f"{file_path}\\2d_combined_data.pkl"
dft_train_features_file_path = f"{file_path}\\bandgap_results\\df_train_bandgap_engineered.pkl"
dft_test_features_file_path = f"{file_path}\\bandgap_results\\df_test_bandgap_engineered.pkl"

# Load feature list and model
df_features_list = joblib.load(features_list_file_path)

model = joblib.load(model_file_path)
# # view LGBMRegressor model
# print(model) 

# Load experimental data, use only 'formula', 'band_gap_dir', and 'doi' columns
df_expt = pd.read_excel(expt_data_file_path)
df_expt = df_expt[['formula', 'band_gap_dir','doi']]

# Load DFT data
df_dft_data = joblib.load(dft_data_file_path)
df_dft_train_features = joblib.load(dft_train_features_file_path)
df_dft_test_features = joblib.load(dft_test_features_file_path)

# combine train and test features into a single dataframe and keep the original indexes
df_dft_features = pd.concat([df_dft_train_features, df_dft_test_features], axis=0)

# # check all indices in df_dft_data are in df_dft_features, print those that are not, and print summary statement
# for idx in df_dft_data.index:
#     if idx not in df_dft_features.index:
#         print(f"Index {idx} not found in df_dft_features")
# print("All indices in df_dft_data are present in df_dft_features")

# if all indices are present, merge the two dataframes on their indices
df_dft_merged = df_dft_data.merge(df_dft_features, left_index=True, right_index=True) 

# keep only 'formula' and the features from df_features_list in df_dft_merged
df_dft_merged = df_dft_merged[['formula'] + df_features_list.tolist()]

# # if the formula for a row in df_expt is in df_dft_merged, print the features in a new df_expt_features dataframe
# df_expt_features = pd.DataFrame()
# for idx, row in df_expt.iterrows():
#     formula = row['formula']
#     if formula in df_dft_merged['formula'].values:
#         features_row = df_dft_merged[df_dft_merged['formula'] == formula]
#         df_expt_features = pd.concat([df_expt_features, features_row], ignore_index=True)

# # Now df_expt_features contains the features for experimental formulas found in the DFT dataset, drop duplicate rows
# df_expt_features = df_expt_features.drop_duplicates().reset_index(drop=True)

# # Figure out which formulas in df_expt are not in df_expt_features and print them to df_formulas_not_found with the formula and the experimental bandgap
# formulas_not_found = []
# for idx, row in df_expt.iterrows():
#     formula = row['formula']
#     if formula not in df_expt_features['formula'].values:
#         formulas_not_found.append({'formula': formula, 'bandgap': row['band_gap_dir']})
# df_formulas_not_found = pd.DataFrame(formulas_not_found)
# print(df_formulas_not_found)


In [ ]:
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty, ElementFraction, Stoichiometry, IonProperty, BandCenter, ValenceOrbital, AtomicOrbitals, ElectronAffinity, ElectronegativityDiff, TMetalFraction, OxidationStates
from pathlib import Path

composition = df_expt['formula']

composition_obj = [Composition.from_dict(item) for item in composition]

df_elemental = pd.DataFrame()
df_compositional = pd.DataFrame()

# Data Sources for ElementProperties
elemental_data_source = ['magpie', 'matminer', 'deml', 'megnet_el']

for idx, des in enumerate(elemental_data_source):
    #if the file already exists, skip
    elemental_feature_descriptors_file_name = fr"{final_file_path}/expt/expt_ElementProperty_{elemental_data_source[idx]}.pkl"
    
    if Path(elemental_feature_descriptors_file_name).exists():
        print(f"File for ElementProperty_{des} already exists. Skipping...")
                
    else:
        ep = ElementProperty.from_preset(des)
        print('Generating element property using ' + des)
        print('No. of features:', len(ep.feature_labels()))
        elemental_features = ep.featurize_many(composition_obj, ignore_errors=True)
        elemental_column_names = ep.feature_labels()
        df_elemental = pd.DataFrame(elemental_features, columns=elemental_column_names)
        # elemental_feature_descriptors_file_name = fr"{features_file_path}/dielectric_ElementProperty_{elemental_data_source[idx]}.pkl"
        joblib.dump(df_elemental, elemental_feature_descriptors_file_name)
        print(df_elemental.head())

# Compositional descriptors
compositional_descriptors_name = ['ElementFraction', 'Stoichiometry', 'IonProperty', 'BandCenter', 'ValenceOrbital', 'AtomicOrbitals', 'ElectronAffinity', 'ElectronegativityDiff', 'TMetalFraction', 'OxidationStates'] # 'IonProperty',
compositional_descriptors = [ElementFraction(), Stoichiometry(), IonProperty(), BandCenter(), ValenceOrbital(), AtomicOrbitals(), ElectronAffinity(), ElectronegativityDiff(), TMetalFraction(), OxidationStates()] # IonProperty()

for idx, des in enumerate(compositional_descriptors):
    compositional_feature_descriptors_file_name = fr"{final_file_path}/expt/expt_{compositional_descriptors_name[idx]}.pkl"

    if Path(compositional_feature_descriptors_file_name).exists():
        print(f"File for {compositional_descriptors_name[idx]} already exists. Skipping...")

    else:
        print('Generating compositional property using ' + compositional_descriptors_name[idx])
        print('No. of features:', len(des.feature_labels()))
        compositional_features = des.featurize_many(composition_obj, ignore_errors=True)
        # compositional_feature_descriptors_file_name = fr"{features_file_path}/dielectric_{compositional_descriptors_name[idx]}.pkl"
        compositional_column_names = des.feature_labels()
        df_compositional = pd.DataFrame(compositional_features, columns=compositional_column_names)
        joblib.dump(df_compositional, compositional_feature_descriptors_file_name)
        print(df_compositional.head())


In [ ]:
column_names = []

for idx, ds in enumerate(elemental_data_source):
    ep = ElementProperty.from_preset(ds)
    column_names.extend(ep.feature_labels())

for idx, des in enumerate(compositional_descriptors):
    column_names.extend(des.feature_labels())

df_expt_features_combined = pd.DataFrame()
df_imported = []

for idx, des in enumerate(elemental_data_source):
    df_file_name = f"{final_file_path}/expt/expt_ElementProperty_{elemental_data_source[idx]}.pkl"
    df_imported.append(joblib.load(df_file_name))

for idx, des in enumerate(compositional_descriptors_name):
    df_file_name = f"{final_file_path}/expt/expt_{compositional_descriptors_name[idx]}.pkl"
    df_imported.append(joblib.load(df_file_name))
    
df_expt_features_combined = pd.concat(df_imported, axis=1)

joblib.dump(df_expt_features_combined, f"{final_file_path}/expt/expt_features_combined.pkl")
    
print("\nCombined Dataframe of Features:\n")
print(df_expt_features_combined.shape)



In [ ]:
# Filter df_expt_features_combined to keep only columns in df_features_list
df_expt_features_filtered = df_expt_features_combined[df_features_list.tolist()]

# add the 'formula' column from df_expt to df_expt_features_filtered
df_expt_features_filtered.insert(0, 'formula', df_expt['formula'])

joblib.dump(df_expt_features_filtered, f"{final_file_path}/expt/expt_features_filtered.pkl")

In [ ]:
# # regenerate the scaler used to scale the training data from df_dft_train_features
# # use minmax scaler
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler(feature_range=(0, 1))
# scaler.fit(df_dft_train_features[df_features_list.tolist()])

# # Use the scaler to scale the features in df_expt_features_filtered (excluding the 'formula' column)
# df_expt_features_scaled = df_expt_features_filtered.copy()
# df_expt_features_scaled[df_features_list.tolist()] = scaler.transform(df_expt_features_filtered[df_features_list.tolist()])
# joblib.dump(df_expt_features_scaled, f"{final_file_path}/expt/expt_features_scaled.pkl")

from sklearn.preprocessing import MinMaxScaler
import numpy as np

# df_dft_features_raw = joblib.load(f"{file_path}\\2d_combined_features_filtered.pkl")
df_dft_features_raw = joblib.load(f"{file_path}\\bandgap_results\\df_train_bandgap.pkl")
df_dft_features_raw = df_dft_features_raw[df_features_list.tolist()]

# Fit scaler to only on columns in the raw features
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df_dft_features_raw)


# Transform experimental data
df_expt_features_scaled = df_expt_features_filtered.copy()
df_expt_features_scaled[df_features_list.tolist()] = scaler.transform(df_expt_features_filtered[df_features_list.tolist()])

joblib.dump(df_expt_features_scaled, f"{final_file_path}/expt/expt_features_scaled.pkl")

# print(df_expt_features_filtered[df_features_list].head())
# print(df_expt_features_filtered[df_features_list].dtypes)
# print(df_dft_train_features[df_features_list].describe())
# print(df_expt_features_scaled[df_features_list].describe())


In [ ]:
# predict bandgaps for df_expt_features using the loaded LGBMRegressor model
expt_bandgap_predictions = model.predict(df_expt_features_scaled[df_features_list.tolist()])

# print the predicted bandgaps alongside the formulas
for i, formula in enumerate(df_expt_features_scaled['formula']):
    print(f"Formula: {formula}, Predicted Band Gap: {expt_bandgap_predictions[i]} eV")

In [ ]:
for col in df_dft_merged.columns:
    p = ks_2samp(df_dft_merged[col], df_expt_features_scaled[col])[1]
    if p < 0.001: # type: ignore
        print(col, "distribution drift detected (p =", p, ")")
    else:
        print(col, "no distribution drift (p =", p, ")")

# save drift results to a text file
with open(f"{final_file_path}\\drift_results_expt_vs_dft.txt", "w") as f:
    for col in df_dft_merged.columns:
        p = ks_2samp(df_dft_merged[col], df_expt_features_scaled[col])[1]
        if p < 0.001: # type: ignore
            f.write(f"{col} distribution drift detected (p = {p})\n")
        else:
            f.write(f"{col} no distribution drift (p = {p})\n")
        

In [ ]:
# Create comparison DataFrame with DFT SOURCES from 'source' column
comparison_df = pd.DataFrame()
comparison_df['formula'] = df_expt_features_scaled['formula']
comparison_df['expt_bandgap'] = df_expt['band_gap_dir'] # [df_expt[df_expt['formula'] == f]['band_gap_dir'].values[0] if f in df_expt['formula'].values else None for f in df_expt_features_scaled['formula']]
def get_dft_bandgap(formula, source):
    match = df_dft_data[(df_dft_data['formula'] == formula) & (df_dft_data['source'] == source)]
    if not match.empty:
        return match['bandgap'].values[0]
    else:
        return None
comparison_df['2dmatpedia'] = [get_dft_bandgap(f, '2dmatpedia') for f in df_expt_features_scaled['formula']]
comparison_df['c2db'] = [get_dft_bandgap(f, 'c2db') for f in df_expt_features_scaled['formula']]
comparison_df['alexandria_2d'] = [get_dft_bandgap(f, 'alexandria_2d') for f in df_expt_features_scaled['formula']]
comparison_df['mc2d'] = [get_dft_bandgap(f, 'mc2d') for f in df_expt_features_scaled['formula']]
comparison_df['gbfs_pred'] = expt_bandgap_predictions
comparison_df['doi'] = df_expt['doi']
print(comparison_df)

# save to pkl and csv
comparison_df.to_pickle(f"{final_file_path}\\expt\\expt_band_gap_dir_pred.pkl")
comparison_df.to_csv(f"{final_file_path}\\expt\\expt_band_gap_dir_pred.csv", index=False)

In [ ]:
# ---------------------------
# Update Matplotlib Parameters
# ---------------------------
plt.rcParams.update({'font.size': 24, 
                    'figure.figsize': (10, 10), 
                    'figure.dpi': 100, 
                    'font.sans-serif': 'Arial',
                    'axes.linewidth': 2.0,
                    'lines.linewidth': 3.0,
                    'xtick.major.width': 2.0,
                    'xtick.minor.visible': True,
                    'xtick.minor.width': 2,
                    'xtick.top': True,
                    'xtick.direction': 'in',
                    'xtick.major.size': 10.0,
                    'xtick.minor.size': 4.0,
                    'xtick.minor.ndivs': 2,
                    'ytick.major.width': 2.0,
                    'ytick.minor.visible': True,
                    'ytick.minor.width': 2,
                    'ytick.right': True,
                    'ytick.direction': 'in',
                    'ytick.major.size': 10.0,
                    'ytick.minor.size': 4.0,
                    'ytick.minor.ndivs': 2
                    })

In [ ]:
# Plot comparison of experimental and GBFS predicted bandgaps without formula labels
plt.figure(figsize=(10, 10))
plt.scatter(comparison_df['expt_bandgap'], comparison_df['gbfs_pred'], color='blue', label='GBFS', s=100)
plt.plot([0, 6], [0, 6], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Band Gaps', fontsize=16)

#add line of best fit and give equation of line
m, b = np.polyfit(comparison_df['expt_bandgap'], comparison_df['gbfs_pred'], 1)
plt.plot(comparison_df['expt_bandgap'], m*comparison_df['expt_bandgap'] + b, color='green', linestyle='-', label=f'Lin. Fit')
plt.text(0, 5.8, f'R²: {r2_score(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f}')
plt.text(0, 5.4, f'RMSE: {root_mean_squared_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 5.0, f'MAE: {mean_absolute_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 4.6, f'N: {len(comparison_df)}')
plt.text(0, 4.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

# Show legend in lower right corner
plt.legend(loc='lower right')

#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap.pdf", dpi=300, bbox_inches='tight')
plt.show()

# output comparison_df to a latex table in longtable format and with \hline after each entry
# restrict to three decimal places and replace all NaN with '-'
# write to a .tex file

# with open(f"{final_file_path}\\expt\\comparison_expt_dft_gbfs.tex", "w") as f:
#     f.write(comparison_df.to_latex(index=False, longtable=True, float_format="%.3f", na_rep='-', header=True, bold_rows=False).replace("\\toprule", "\\hline").replace("\\midrule", "\\hline").replace("\\bottomrule", "\\hline").replace("\\endhead", "\\hline\n"))

In [ ]:
# Plot comparison of experimental and GBFS predicted bandgaps with formula labels
plt.figure(figsize=(10, 10))
plt.scatter(comparison_df['expt_bandgap'], comparison_df['gbfs_pred'], color='blue', label='GBFS', s=100)


for i, row in comparison_df.iterrows():
    plt.text(row['expt_bandgap'], row['gbfs_pred'], str(row['formula']), fontsize=8, ha='right', va='bottom', alpha=0.7)

# texts = []
# for _, row in comparison_df.iterrows():
#     texts.append(
#         plt.text(
#             row['expt_bandgap'],
#             row['gbfs_pred'],
#             row['formula'],
#             fontsize=10,
#             path_effects=[pe.withStroke(linewidth=2, foreground="white")],
#             ha='right', va='bottom', alpha=0.7
#         )
#     )

# adjust_text(
#     texts,
#     arrowprops=dict(arrowstyle='-', color='grey', lw=0.5),
#     expand_points=(20, 20),
#     expand_text=(20, 20),
#     force_text=0.8,
#     force_points=0.5
# )


plt.plot([0, 6], [0, 6], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Band Gaps', fontsize=16)

#add line of best fit and give equation of line
m, b = np.polyfit(comparison_df['expt_bandgap'], comparison_df['gbfs_pred'], 1)
plt.plot(comparison_df['expt_bandgap'], m*comparison_df['expt_bandgap'] + b, color='green', linestyle='-', label=f'Lin. Fit')
plt.text(0, 5.8, f'R²: {r2_score(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f}')
plt.text(0, 5.4, f'RMSE: {root_mean_squared_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 5.0, f'MAE: {mean_absolute_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 4.6, f'N: {len(comparison_df)}')
plt.text(0, 4.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

# Show legend in lower right corner
plt.legend(loc='lower right')

#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_with_labels.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_with_labels.pdf", dpi=300, bbox_inches='tight')
plt.show()

   


In [ ]:
# define chemical families and their colours
chemical_families = {
    'Oxides': {'elements': ['O'], 'color': 'red', 'marker': 'o'},
    'Phosphides': {'elements': ['P'], 'color': 'purple', 'marker': 'D'},
    'Halides': {'elements': ['F', 'Cl', 'Br', 'I'], 'color': 'green', 'marker': 's'},
    'Nitrides': {'elements': ['N'], 'color': 'orange', 'marker': '^'},
    'Chalcogenides': {'elements': ['S', 'Se', 'Te'], 'color': 'blue', 'marker': 'v'},
    'Others': {'elements': [], 'color': 'grey', 'marker': 'X'}
}
# function to determine chemical family based on formula using pymatgen Composition to parse the formula and check for the presence of elements in each family
# allow for multiple families to be detected
def determine_family(formula):
    comp = Composition(formula)
    elements = [el.symbol for el in comp.elements]
    families = []
    for family, data in chemical_families.items():
        if any(el in data['elements'] for el in elements):
            families.append(family)
    return families if families else ['Others']

In [ ]:
# replicate the GBFS expt vs dft plot with different colours for chemical families

# add chemical family column to comparison_df
comparison_df['chemical_family'] = comparison_df['formula'].apply(determine_family)

# plot with different colours for chemical families, where there are multiple families for each formula, plot them both
plt.figure(figsize=(10, 10))
for i, row in comparison_df.iterrows():
    families = row['chemical_family']
    if not isinstance(families, list):
        families = [families]
    for family in families:
        data = chemical_families[family]
        plt.scatter(row['expt_bandgap'], row['gbfs_pred'], color=data['color'], label=family, s=100, alpha=0.7, marker=data['marker'])
# Remove duplicate labels in legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.plot([0, 6], [0, 6], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Bandgaps by Chemical Family', fontsize=16)
#add line of best fit and give equation of line
m, b = np.polyfit(comparison_df['expt_bandgap'], comparison_df['gbfs_pred'], 1)
plt.plot(comparison_df['expt_bandgap'], m*comparison_df['expt_bandgap'] + b, color='black', linestyle='-', label=f'Lin. Fit')
plt.text(0, 5.8, f'R²: {r2_score(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f}')
plt.text(0, 5.4, f'RMSE: {root_mean_squared_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 5.0, f'MAE: {mean_absolute_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 4.6, f'N: {len(comparison_df)}')
plt.text(0, 4.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

# Show legend in lower right corner
plt.legend( by_label.values(), by_label.keys(), loc='lower right')

#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family.pdf", dpi=300, bbox_inches='tight')# replicate the GBFS expt vs dft plot with different colours for chemical families

In [ ]:
# replicate the GBFS expt vs dft plot with different colours for chemical families

# add chemical family column to comparison_df
comparison_df['chemical_family'] = comparison_df['formula'].apply(determine_family)

# plot with different colours for chemical families, where there are multiple families for each formula, plot them both
plt.figure(figsize=(10, 10))
for i, row in comparison_df.iterrows():
    families = row['chemical_family']
    if not isinstance(families, list):
        families = [families]
    for family in families:
        data = chemical_families[family]
        plt.scatter(row['expt_bandgap'], row['gbfs_pred'], color=data['color'], label=family, s=100, alpha=0.7, marker=data['marker'])
        plt.text(row['expt_bandgap'], row['gbfs_pred'], str(row['formula']), fontsize=10, ha='right', va='bottom', alpha=0.7, color=data['color'])

# Remove duplicate labels in legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.plot([0, 6], [0, 6], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Bandgaps by Chemical Family', fontsize=16)
#add line of best fit and give equation of line
m, b = np.polyfit(comparison_df['expt_bandgap'], comparison_df['gbfs_pred'], 1)
plt.plot(comparison_df['expt_bandgap'], m*comparison_df['expt_bandgap'] + b, color='black', linestyle='-', label=f'Lin. Fit')
plt.text(0, 5.8, f'R²: {r2_score(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f}')
plt.text(0, 5.4, f'RMSE: {root_mean_squared_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 5.0, f'MAE: {mean_absolute_error(comparison_df["expt_bandgap"], comparison_df["gbfs_pred"]):.3f} eV')
plt.text(0, 4.6, f'N: {len(comparison_df)}')
plt.text(0, 4.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

# Show legend in lower right corner

plt.legend(by_label.values(), by_label.keys(), loc='lower right')
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family_with_labels.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family_with_labels.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot each chemical family separately in its own subplot of a 2x3 grid
fig, axs = plt.subplots(2, 3, figsize=(20, 15))
for ax, (family, data) in zip(axs.flatten(), chemical_families.items()):
    # Plot all materials belonging to this family, including those in multiple families
    for i, row in comparison_df.iterrows():
        families = row['chemical_family']
        if not isinstance(families, list):
            families = [families]
        if family in families:
            ax.scatter(row['expt_bandgap'], row['gbfs_pred'], color=data['color'], label=family, s=100, marker=data['marker'])
    # Prepare subset for stats and fit
    subset = comparison_df[comparison_df['chemical_family'].apply(lambda fams: family in fams if isinstance(fams, list) else fams == family)]
    max_val =  math.ceil(max(subset['expt_bandgap'].max(), subset['gbfs_pred'].max()) if not subset.empty else 7)
    ax.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='y=x Line')
    ax.set_xlabel('Experimental Band Gap [eV]')
    ax.set_ylabel('GBFS Predicted Band Gap [eV]')
    ax.set_xlim(0, max_val+0.5)
    ax.set_ylim(0, max_val+0.5)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))
    ax.set_title(f'{family}')
    if len(subset) > 5:
        m, b = np.polyfit(subset['expt_bandgap'], subset['gbfs_pred'], 1)
        ax.plot(subset['expt_bandgap'], m*subset['expt_bandgap'] + b, color='black', linestyle='-', label='Lin. Fit')
        ax.text(max_val*0.05, max_val, f'R²: {r2_score(subset["expt_bandgap"], subset["gbfs_pred"]):.3f}')
        ax.text(max_val*0.05, max_val*0.94, f'RMSE: {root_mean_squared_error(subset["expt_bandgap"], subset["gbfs_pred"]):.3f} eV')
        ax.text(max_val*0.05, max_val*0.88, f'MAE: {mean_absolute_error(subset["expt_bandgap"], subset["gbfs_pred"]):.3f} eV')
        ax.text(max_val*0.05, max_val*0.82, f'N: {len(subset)}')
        ax.text(max_val*0.05, max_val*0.76, f'Lin. Fit: y={m:.2f}x+{b:.2f}')
    else:
        ax.text(max_val*0.05, max_val, f'N: {len(subset)}')
    # ax.legend( by_label.values(), by_label.keys(), loc='lower right', fontsize=16)

# Adjust layout
plt.tight_layout()
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family_subplots.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family_subplots.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot each chemical family separately in its own subplot of a 2x3 grid
fig, axs = plt.subplots(2, 3, figsize=(20, 15))
for ax, (family, data) in zip(axs.flatten(), chemical_families.items()):
    # Plot all materials belonging to this family, including those in multiple families
    for i, row in comparison_df.iterrows():
        families = row['chemical_family']
        if not isinstance(families, list):
            families = [families]
        if family in families:
            ax.scatter(row['expt_bandgap'], row['gbfs_pred'], color=data['color'], label=family, s=100, marker=data['marker'])
            ax.text(row['expt_bandgap'], row['gbfs_pred'], str(row['formula']), fontsize=12, ha='right', va='bottom', alpha=0.7, color=data['color'])
    # Prepare subset for stats and fit
    subset = comparison_df[comparison_df['chemical_family'].apply(lambda fams: family in fams if isinstance(fams, list) else fams == family)]
    max_val =  math.ceil(max(subset['expt_bandgap'].max(), subset['gbfs_pred'].max()) if not subset.empty else 7)
    ax.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='y=x Line')
    ax.set_xlabel('Experimental Band Gap [eV]')
    ax.set_ylabel('GBFS Predicted Band Gap [eV]')
    ax.set_xlim(-0.5, max_val+0.5)
    ax.set_ylim(-0.5, max_val+0.5)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))
    ax.set_title(f'{family}')
    if len(subset) > 1:
        m, b = np.polyfit(subset['expt_bandgap'], subset['gbfs_pred'], 1)
        ax.plot(subset['expt_bandgap'], m*subset['expt_bandgap'] + b, color='black', linestyle='-', label='Lin. Fit')
        ax.text(-0.25, max_val, f'R²: {r2_score(subset["expt_bandgap"], subset["gbfs_pred"]):.3f}')
        ax.text(-0.25, max_val*0.94, f'RMSE: {root_mean_squared_error(subset["expt_bandgap"], subset["gbfs_pred"]):.3f} eV')
        ax.text(-0.25, max_val*0.88, f'MAE: {mean_absolute_error(subset["expt_bandgap"], subset["gbfs_pred"]):.3f} eV')
        ax.text(-0.25, max_val*0.82, f'N: {len(subset)}')
        ax.text(-0.25, max_val*0.76, f'Lin. Fit: y={m:.2f}x+{b:.2f}')
    else:
        ax.text(-0.25, max_val, f'N: {len(subset)}')
    # ax.legend( by_label.values(), by_label.keys(), loc='lower right', fontsize=16)


# Adjust layout
plt.tight_layout()
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family_subplots_with_labels.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_by_chemical_family_subplots_with_labels.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
chemical_family_to_plot = 'Chalcogenides'

#plot just the chalchogenides with formula labels
plt.figure(figsize=(10, 10))
# collect all entries that have chemical_family_to_plot
subset = comparison_df[comparison_df['chemical_family'].apply(lambda fams: chemical_family_to_plot in fams if isinstance(fams, list) else fams == chemical_family_to_plot)]
plt.scatter(subset['expt_bandgap'], subset['gbfs_pred'], color=chemical_families[chemical_family_to_plot]['color'], label=chemical_family_to_plot, s=100, alpha=0.7, marker='v')
for i, row in subset.iterrows():
    plt.text(row['expt_bandgap'], row['gbfs_pred'], str(row['formula']), fontsize=10, ha='right', va='bottom', color='black')
plt.plot([0, 4], [0, 4], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Bandgaps for Chalcogenides', fontsize=16)
#add line of best fit and give equation of line
m, b = np.polyfit(subset['expt_bandgap'], subset['gbfs_pred'], 1)
plt.plot(subset['expt_bandgap'], m*subset['expt_bandgap'] + b, color='green', linestyle='-', label=f'Lin. Fit')
plt.text(0, 3.8, f'R²: {r2_score(subset["expt_bandgap"], subset["gbfs_pred"]):.3f}')
plt.text(0, 3.5, f'RMSE: {root_mean_squared_error(subset["expt_bandgap"], subset["gbfs_pred"]):.3f} eV')
plt.text(0, 3.2, f'MAE: {mean_absolute_error(subset["expt_bandgap"], subset["gbfs_pred"]):.3f} eV')
plt.text(0, 2.9, f'N: {len(subset)}')
plt.text(0, 2.6, f'Lin. Fit: y={m:.3f}x+{b:.3f}')
# Show legend in lower right corner
plt.legend(loc='lower right', fontsize=12)
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_{chemical_family_to_plot}_with_labels.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_{chemical_family_to_plot}_with_labels.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot comparison of experimental and Alexandria DFT predicted bandgaps

valid_entries_alexandria_2d = comparison_df.dropna(subset=['expt_bandgap', 'alexandria_2d'])

plt.figure(figsize=(10, 10))
plt.scatter(valid_entries_alexandria_2d['expt_bandgap'], valid_entries_alexandria_2d['alexandria_2d'], color='blue', label='Alexandria_2D', s=100)

plt.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('Alexandria_2D Predicted Band Gap [eV]')

# Add line of best fit and give equation of line
m, b = np.polyfit(valid_entries_alexandria_2d['expt_bandgap'], valid_entries_alexandria_2d['alexandria_2d'], 1)
plt.plot(valid_entries_alexandria_2d['expt_bandgap'], m*valid_entries_alexandria_2d['expt_bandgap'] + b, color='green', linestyle='-', label='Lin. Fit')

plt.text(0, 6.8, f'R²: {r2_score(valid_entries_alexandria_2d["expt_bandgap"], valid_entries_alexandria_2d["alexandria_2d"]):.3f}')
plt.text(0, 6.4, f'RMSE: {root_mean_squared_error(valid_entries_alexandria_2d["expt_bandgap"], valid_entries_alexandria_2d["alexandria_2d"]):.3f} eV')
plt.text(0, 6.0, f'MAE: {mean_absolute_error(valid_entries_alexandria_2d["expt_bandgap"], valid_entries_alexandria_2d["alexandria_2d"]):.3f} eV')
plt.text(0, 5.6, f'N: {len(valid_entries_alexandria_2d)}')
plt.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

plt.legend(loc='lower right')
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_alexandria_2d_bandgap.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_alexandria_2d_bandgap.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Plot comparison of experimental and Alexandria DFT predicted bandgaps

valid_entries_alexandria_2d = comparison_df.dropna(subset=['expt_bandgap', 'alexandria_2d'])

plt.figure(figsize=(10, 10))
plt.scatter(valid_entries_alexandria_2d['expt_bandgap'], valid_entries_alexandria_2d['alexandria_2d'], color='blue', label='Alexandria_2D', s=100)
for i, row in valid_entries_alexandria_2d.iterrows():
    plt.text(row['expt_bandgap'], row['alexandria_2d'], str(row['formula']), fontsize=10, ha='right', va='bottom', color='black')

plt.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('Alexandria_2D Predicted Band Gap [eV]')

# Add line of best fit and give equation of line
m, b = np.polyfit(valid_entries_alexandria_2d['expt_bandgap'], valid_entries_alexandria_2d['alexandria_2d'], 1)
plt.plot(valid_entries_alexandria_2d['expt_bandgap'], m*valid_entries_alexandria_2d['expt_bandgap'] + b, color='green', linestyle='-', label='Lin. Fit')

plt.text(0, 6.8, f'R²: {r2_score(valid_entries_alexandria_2d["expt_bandgap"], valid_entries_alexandria_2d["alexandria_2d"]):.3f}')
plt.text(0, 6.4, f'RMSE: {root_mean_squared_error(valid_entries_alexandria_2d["expt_bandgap"], valid_entries_alexandria_2d["alexandria_2d"]):.3f} eV')
plt.text(0, 6.0, f'MAE: {mean_absolute_error(valid_entries_alexandria_2d["expt_bandgap"], valid_entries_alexandria_2d["alexandria_2d"]):.3f} eV')
plt.text(0, 5.6, f'N: {len(valid_entries_alexandria_2d)}')
plt.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

plt.legend(loc='lower right')
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_alexandria_2d_bandgap_with_labels.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_alexandria_2d_bandgap_with_labels.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# plot each chemical family separately in its own subplot of a 2x3 grid
fig, axs = plt.subplots(2, 3, figsize=(20, 15))
for ax, (family, data) in zip(axs.flatten(), chemical_families.items()):
    # Plot all materials belonging to this family, including those in multiple families
    for i, row in valid_entries_alexandria_2d.iterrows():
        families = row['chemical_family']
        if not isinstance(families, list):
            families = [families]
        if family in families:
            ax.scatter(row['expt_bandgap'], row['alexandria_2d'], color=data['color'], label=family, s=100, marker=data['marker'])
            ax.text(row['expt_bandgap'], row['alexandria_2d'], str(row['formula']), fontsize=12, ha='right', va='bottom', alpha=0.7, color=data['color'])
    # Prepare subset for stats and fit
    subset = valid_entries_alexandria_2d[valid_entries_alexandria_2d['chemical_family'].apply(lambda fams: family in fams if isinstance(fams, list) else fams == family)]
    ax.set_xlabel('Experimental Band Gap [eV]')
    ax.set_ylabel('Alexandria_2D Predicted Band Gap [eV]')

    max_val =  math.ceil(max(subset['expt_bandgap'].max(), subset['alexandria_2d'].max()) if not subset.empty else 7)
    ax.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='y=x Line')
    ax.set_xlabel('Experimental Band Gap [eV]')
    ax.set_ylabel('Alexandria_2D Predicted Band Gap [eV]')
    ax.set_xlim(-0.5, max_val+0.5)
    ax.set_ylim(-0.5, max_val+0.5)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))

    ax.set_title(f'{family}')
    if len(subset) > 1:
        m, b = np.polyfit(subset['expt_bandgap'], subset['alexandria_2d'], 1)
        ax.plot(subset['expt_bandgap'], m*subset['expt_bandgap'] + b, color='black', linestyle='-', label='Lin. Fit')
        ax.text(-0.25, max_val, f'R²: {r2_score(subset["expt_bandgap"], subset["alexandria_2d"]):.3f}')
        ax.text(-0.25, max_val*0.94, f'RMSE: {root_mean_squared_error(subset["expt_bandgap"], subset["alexandria_2d"]):.3f} eV')
        ax.text(-0.25, max_val*0.88, f'MAE: {mean_absolute_error(subset["expt_bandgap"], subset["alexandria_2d"]):.3f} eV')
        ax.text(-0.25, max_val*0.82, f'N: {len(subset)}')
        ax.text(-0.25, max_val*0.76, f'Lin. Fit: y={m:.2f}x+{b:.2f}')
    else:
        ax.text(-0.25, max_val, f'N: {len(subset)}')
    # ax.legend( by_label.values(), by_label.keys(), loc='lower right', fontsize=16)


# Adjust layout
plt.tight_layout()
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_alexandria_2d_bandgap_by_chemical_family_subplots.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_alexandria_2d_bandgap_by_chemical_family_subplots.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot comparison of experimental and 2dmatpedia DFT predicted bandgaps

valid_entries_2dmatpedia = comparison_df.dropna(subset=['expt_bandgap', '2dmatpedia'])

plt.figure(figsize=(10, 10))
plt.scatter(valid_entries_2dmatpedia['expt_bandgap'], valid_entries_2dmatpedia['2dmatpedia'], color='blue', label='2dmatpedia', s=100)
plt.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('2dmatpedia Predicted Band Gap [eV]')

# Add line of best fit and give equation of line
m, b = np.polyfit(valid_entries_2dmatpedia['expt_bandgap'], valid_entries_2dmatpedia['2dmatpedia'], 1)
plt.plot(valid_entries_2dmatpedia['expt_bandgap'], m*valid_entries_2dmatpedia['expt_bandgap'] + b, color='green', linestyle='-', label='Lin. Fit')

plt.text(0, 6.8, f'R²: {r2_score(valid_entries_2dmatpedia["expt_bandgap"], valid_entries_2dmatpedia["2dmatpedia"]):.3f}')
plt.text(0, 6.4, f'RMSE: {root_mean_squared_error(valid_entries_2dmatpedia["expt_bandgap"], valid_entries_2dmatpedia["2dmatpedia"]):.3f} eV')
plt.text(0, 6.0, f'MAE: {mean_absolute_error(valid_entries_2dmatpedia["expt_bandgap"], valid_entries_2dmatpedia["2dmatpedia"]):.3f} eV')
plt.text(0, 5.6, f'N: {len(valid_entries_2dmatpedia)}')
plt.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

plt.legend(loc='lower right')
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_2dmatpedia_bandgap.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_2dmatpedia_bandgap.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Plot comparison of experimental and c2db DFT predicted bandgaps

valid_entries_c2db = comparison_df.dropna(subset=['expt_bandgap', 'c2db'])

plt.figure(figsize=(10, 10))
plt.scatter(valid_entries_c2db['expt_bandgap'], valid_entries_c2db['c2db'], color='blue', label='c2db', s=100)
plt.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('c2db Predicted Band Gap [eV]')

# Add line of best fit and give equation of line
m, b = np.polyfit(valid_entries_c2db['expt_bandgap'], valid_entries_c2db['c2db'], 1)
plt.plot(valid_entries_c2db['expt_bandgap'], m*valid_entries_c2db['expt_bandgap'] + b, color='green', linestyle='-', label='Lin. Fit')

plt.text(0, 6.8, f'R²: {r2_score(valid_entries_c2db["expt_bandgap"], valid_entries_c2db["c2db"]):.3f}')
plt.text(0, 6.4, f'RMSE: {root_mean_squared_error(valid_entries_c2db["expt_bandgap"], valid_entries_c2db["c2db"]):.3f} eV')
plt.text(0, 6.0, f'MAE: {mean_absolute_error(valid_entries_c2db["expt_bandgap"], valid_entries_c2db["c2db"]):.3f} eV')
plt.text(0, 5.6, f'N: {len(valid_entries_c2db)}')
plt.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

plt.legend(loc='lower right')
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_c2db_bandgap.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_c2db_bandgap.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Plot comparison of experimental and mc2d DFT predicted bandgaps

valid_entries_mc2d = comparison_df.dropna(subset=['expt_bandgap', 'mc2d'])

plt.figure(figsize=(10, 10))
plt.scatter(valid_entries_mc2d['expt_bandgap'], valid_entries_mc2d['mc2d'], color='blue', label='mc2d', s=100)
plt.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('mc2d Predicted Band Gap [eV]')

# Add line of best fit and give equation of line
m, b = np.polyfit(valid_entries_mc2d['expt_bandgap'], valid_entries_mc2d['mc2d'], 1)
plt.plot(valid_entries_mc2d['expt_bandgap'], m*valid_entries_mc2d['expt_bandgap'] + b, color='green', linestyle='-', label='Lin. Fit')

plt.text(0, 6.8, f'R²: {r2_score(valid_entries_mc2d["expt_bandgap"], valid_entries_mc2d["mc2d"]):.3f}')
plt.text(0, 6.4, f'RMSE: {root_mean_squared_error(valid_entries_mc2d["expt_bandgap"], valid_entries_mc2d["mc2d"]):.3f} eV')
plt.text(0, 6.0, f'MAE: {mean_absolute_error(valid_entries_mc2d["expt_bandgap"], valid_entries_mc2d["mc2d"]):.3f} eV')
plt.text(0, 5.6, f'N: {len(valid_entries_mc2d)}')
plt.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

plt.legend(loc='lower right')
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_mc2d_bandgap.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_mc2d_bandgap.pdf", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# plot comparison of union of DFT predicted bandgaps vs experimental bandgaps


In [ ]:
#combine all four DFT sub-plots into a single figure with 2x2 subplots by importing the saved png files
fig, axs = plt.subplots(2, 2, figsize=(20, 20))

for i, source in enumerate(DFT_SOURCES):
    img = plt.imread(f"{final_file_path}\\expt\\expt_vs_{source}_bandgap.png")
    ax = axs[i//2, i%2]
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(source, fontsize=24)
plt.tight_layout()
#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_dft_bandgap_combined.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_dft_bandgap_combined.pdf", dpi=300, bbox_inches='tight')


In [ ]:
# Compute residuals and fit correction models for DFT SOURCES and GBFS predictions
from sklearn.linear_model import LinearRegression

corrected_predictions = {}

for source in SOURCES:
    valid_entries = comparison_df.dropna(subset=['expt_bandgap', source])
    residuals = valid_entries['expt_bandgap'] - valid_entries[source]
    # Fit a linear regression model to predict residuals from predicted values
    X = valid_entries[[source]].values
    y = residuals.values
    reg = LinearRegression().fit(X, y)
    # Apply correction: corrected_pred = original_pred + predicted_residual
    predicted_residuals = reg.predict(comparison_df[[source]].fillna(0).values)
    corrected_predictions[source] = comparison_df[source] + predicted_residuals
    print(f"{source} correction: y = {reg.coef_[0]:.3f}*x + {reg.intercept_:.3f}")

# Add corrected predictions to comparison_df
for source in SOURCES:
    comparison_df[f'{source}_corrected'] = corrected_predictions[source]

print(comparison_df[[
    'formula', 'expt_bandgap',
    'gbfs_pred', 'gbfs_pred_corrected',
    'alexandria_2d', 'alexandria_2d_corrected',
    '2dmatpedia', '2dmatpedia_corrected',
    'c2db', 'c2db_corrected',
    'mc2d', 'mc2d_corrected'
]])

In [ ]:
# Plot corrected predictions vs experimental values for each source in a 2x3 grid of subplots, with the sixth subplot as the union of DFT techniques
fig, axs = plt.subplots(2, 3, figsize=(30, 20))
for i, source in enumerate(SOURCES):
    corrected_col = f'{source}_corrected'
    valid_entries = comparison_df.dropna(subset=['expt_bandgap', corrected_col])
    ax = axs[i//3, i%3]
    ax.scatter(valid_entries['expt_bandgap'], valid_entries[corrected_col], color='blue', label=f'{source}', s=100)
    ax.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
    ax.set_xlabel('Experimental Band Gap [eV]')
    ax.set_ylabel(f'Adjusted Band Gap {source} [eV]')
    m, b = np.polyfit(valid_entries['expt_bandgap'], valid_entries[corrected_col], 1)
    ax.plot(valid_entries['expt_bandgap'], m*valid_entries['expt_bandgap'] + b, color='green', linestyle='-', label='Lin. Fit')
    ax.text(0, 6.8, f'R²: {np.corrcoef(valid_entries["expt_bandgap"], valid_entries[corrected_col])[0,1]**2:.3f}')
    ax.text(0, 6.4, f'RMSE: {np.sqrt(np.mean((valid_entries["expt_bandgap"] - valid_entries[corrected_col])**2)):.3f} eV')
    ax.text(0, 6.0, f'MAE: {np.mean(np.abs(valid_entries["expt_bandgap"] - valid_entries[corrected_col])):.3f} eV')
    ax.text(0, 5.6, f'N: {len(valid_entries)}')
    ax.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')
    ax.legend(loc='lower right')
    ax.set_title(f'{source} (lin. reg.)', fontsize=24)
# Sixth subplot: union of DFT techniques
ax_union = axs[1,2]
for source in DFT_SOURCES:
    corrected_col = f'{source}_corrected'
    valid_entries = comparison_df.dropna(subset=['expt_bandgap', corrected_col])
    ax_union.scatter(valid_entries['expt_bandgap'], valid_entries[corrected_col], label=source, s=100, alpha=0.7)
    m, b = np.polyfit(valid_entries['expt_bandgap'], valid_entries[corrected_col], 1)
    ax_union.plot(valid_entries['expt_bandgap'], m*valid_entries['expt_bandgap'] + b, linestyle='-')
ax_union.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
ax_union.set_xlabel('Experimental Band Gap [eV]')
ax_union.set_ylabel('Corrected DFT Predicted Band Gap [eV]')
ax_union.set_title('Union of Corrected DFT Techniques', fontsize=24)
ax_union.legend()
plt.tight_layout()
plt.show()

In [ ]:
predictor_dict = {
    'gbfs_pred': {'color': 'black', 'marker': 'o', 'linestyle': '-'},
    'alexandria_2d': {'color': 'purple', 'marker': 'D', 'linestyle': '--'},
    '2dmatpedia': {'color': 'green', 'marker': 's', 'linestyle': '-.'},
    'c2db': {'color': 'orange', 'marker': '^', 'linestyle': ':'},
    'mc2d': {'color': 'blue', 'marker': 'v', 'linestyle': '-'}
}

# Plot all predictions vs experimental values for each source in a 2x3 grid of subplots, with the sixth subplot as the union of DFT techniques
# add (a), (b), (c) labels to each subplot
fig, axs = plt.subplots(2, 3, figsize=(30, 20))
subplot_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
for i, source in enumerate(SOURCES):
    # corrected_col = f'{source}_corrected'
    valid_entries = comparison_df.dropna(subset=['expt_bandgap', source])
    ax = axs[i//3, i%3]
    ax.scatter(valid_entries['expt_bandgap'], valid_entries[source], color=predictor_dict[source]['color'], marker=predictor_dict[source]['marker'], label=f'{source}', s=100, alpha=0.7)
    ax.plot([0, 7], [0, 7], color='red', linestyle='--', label='y=x Line')
    ax.set_xlabel('Experimental Band Gap [eV]')
    ax.set_ylabel(f'Band Gap from {source} [eV]')
    m, b = np.polyfit(valid_entries['expt_bandgap'], valid_entries[source], 1)
    ax.plot(valid_entries['expt_bandgap'], m*valid_entries['expt_bandgap'] + b, linestyle='-', color=predictor_dict[source]['color'], label='Lin. Fit')
    ax.text(0, 6.8, f'R²: {r2_score(valid_entries["expt_bandgap"], valid_entries[source]):.3f}')
    ax.text(0, 6.4, f'RMSE: {root_mean_squared_error(valid_entries["expt_bandgap"], valid_entries[source]):.3f} eV')
    ax.text(0, 6.0, f'MAE: {mean_absolute_error(valid_entries["expt_bandgap"], valid_entries[source]):.3f} eV')
    ax.text(0, 5.6, f'N: {len(valid_entries)}')
    ax.text(0, 5.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')
    ax.legend(loc='lower right')
    ax.set_title(f'{source}', fontsize=24)
    # Add subplot label
    ax.text(-0.05, 1.05, subplot_labels[i], transform=ax.transAxes, fontsize=28, fontweight='bold', va='top', ha='left')
# Sixth subplot: calculate and plot residual histograms for GBFS and each of the DFT techniques 
ax_hist = axs[1,2]
bins = np.linspace(-2.5, 2.5, 50)
for source in SOURCES:
    valid_entries = comparison_df.dropna(subset=['expt_bandgap', source])
    residuals = pd.DataFrame()
    residuals['value'] = valid_entries['expt_bandgap'] -  valid_entries[source]
    ax_hist.hist(residuals['value'], histtype='stepfilled', label=source, alpha=0.5,  bins=bins, color=predictor_dict[source]['color'])
    ax_hist.hist(residuals['value'], histtype='step',  bins=bins, color=predictor_dict[source]['color'], edgecolor=predictor_dict[source]['color'], linewidth=2)
    # sns.kdeplot(residuals['value'], color=predictor_dict[source]['color'], linewidth=2, ax=ax_hist, fill=True)
ax_hist.set_xlabel('Residuals [eV]')
ax_hist.set_ylabel('Frequency')
ax_hist.set_title('Residual Histograms', fontsize=24)
ax_hist.legend()

plt.tight_layout()
# save the figure
plt.savefig(f"{final_file_path}\\expt\\expt_vs_all_bandgap_combined.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_all_bandgap_combined.pdf", dpi=300, bbox_inches='tight')
            
plt.show()

In [ ]:
# Plot comparison of experimental and GBFS predicted bandgaps for rows where DFT_SOURCES are all null

df_expt_only = comparison_df[comparison_df[DFT_SOURCES].isnull().all(axis=1)]

plt.figure(figsize=(10, 10))
plt.scatter(df_expt_only['expt_bandgap'], df_expt_only['gbfs_pred'], color='blue', label='GBFS', s=100)

plt.plot([0, 6], [0, 6], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Band Gaps', fontsize=16)

#add line of best fit and give equation of line
m, b = np.polyfit(df_expt_only['expt_bandgap'], df_expt_only['gbfs_pred'], 1)
plt.plot(df_expt_only['expt_bandgap'], m*df_expt_only['expt_bandgap'] + b, color='green', linestyle='-', label=f'Lin. Fit')
plt.text(0, 5.8, f'R²: {r2_score(df_expt_only["expt_bandgap"], df_expt_only["gbfs_pred"]):.3f}')
plt.text(0, 5.4, f'RMSE: {root_mean_squared_error(df_expt_only["expt_bandgap"], df_expt_only["gbfs_pred"]):.3f} eV')
plt.text(0, 5.0, f'MAE: {mean_absolute_error(df_expt_only["expt_bandgap"], df_expt_only["gbfs_pred"]):.3f} eV')
plt.text(0, 4.6, f'N: {len(df_expt_only)}')
plt.text(0, 4.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

# Show legend in lower right corner
plt.legend(loc='lower right')

#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_expt_only.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_expt_only.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot comparison of experimental and GBFS predicted bandgaps for rows where DFT_SOURCES are all null

df_expt_only = comparison_df[comparison_df[DFT_SOURCES].isnull().all(axis=1)]

plt.figure(figsize=(10, 10))
plt.scatter(df_expt_only['expt_bandgap'], df_expt_only['gbfs_pred'], color='blue', label='GBFS', s=100, alpha=0.7)

texts = []
for _, row in df_expt_only.iterrows():
    texts.append(
        plt.text(
            row['expt_bandgap'],
            row['gbfs_pred'],
            row['formula'],
            fontsize=10,
            path_effects=[pe.withStroke(linewidth=2, foreground="white")]
        )
    )

adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='grey', lw=0.5),
    expand_points=(20, 20),
    expand_text=(20, 20),
    force_text=0.8,
    force_points=0.5
)

# for i, row in df_expt_only.iterrows():
#     plt.text(row['expt_bandgap'], row['gbfs_pred'], str(row['formula']), fontsize=12, ha='right', va='bottom')
plt.plot([0, 6], [0, 6], color='red', linestyle='--', label='y=x Line')
plt.xlabel('Experimental Band Gap [eV]')
plt.ylabel('GBFS Predicted Band Gap [eV]')
# plt.title('Comparison of Experimental and GBFS Predicted Band Gaps', fontsize=16)

#add line of best fit and give equation of line
m, b = np.polyfit(df_expt_only['expt_bandgap'], df_expt_only['gbfs_pred'], 1)
plt.plot(df_expt_only['expt_bandgap'], m*df_expt_only['expt_bandgap'] + b, color='green', linestyle='-', label=f'Lin. Fit')
plt.text(0, 5.8, f'R²: {r2_score(df_expt_only["expt_bandgap"], df_expt_only["gbfs_pred"]):.3f}')
plt.text(0, 5.4, f'RMSE: {root_mean_squared_error(df_expt_only["expt_bandgap"], df_expt_only["gbfs_pred"]):.3f} eV')
plt.text(0, 5.0, f'MAE: {mean_absolute_error(df_expt_only["expt_bandgap"], df_expt_only["gbfs_pred"]):.3f} eV')
plt.text(0, 4.6, f'N: {len(df_expt_only)}')
plt.text(0, 4.2, f'Lin. Fit: y={m:.3f}x+{b:.3f}')

# Show legend in lower right corner
plt.legend(loc='lower right')

#save figure as png and pdf
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_expt_only_with_labels.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{final_file_path}\\expt\\expt_vs_gbfs_bandgap_expt_only_with_labels.pdf", dpi=300, bbox_inches='tight')
plt.show() 